# ADC Example 2 — Scan Mode with DMA

**Target:** STM32F103C8T6 · ADC1 · 3 channels (PA0/IN0, PA1/IN1, PA4/IN4)
**Focus:** Automatic sequential conversion of multiple channels, with DMA moving results to memory.

---
## Why Scan Mode Needs DMA (or Prompt Reading)

The F103 ADC has **one data register (`DR`)** for the entire regular group. In scan mode, Rank 2's result overwrites Rank 1's result in `DR` if the CPU does not read between conversions. DMA solves this by reading `DR` immediately after each rank completes, storing each result in the correct buffer slot[reference:1].

---
## CubeMX Configuration

### Clock
- HCLK = 72 MHz, ADC Prescaler = 6 → ADCCLK = 12 MHz.

### ADC1 Parameters
| Parameter | Setting |
| :--- | :--- |
| `Scan Conversion Mode` | **Enabled** |
| `Continuous Conversion Mode` | Disabled (for single sweep) or Enabled (for circular free-run) |
| `Number Of Conversion` | **3** |
| Rank 1 | Channel 0 (PA0), Sampling Time 55.5 Cycles |
| Rank 2 | Channel 1 (PA1), Sampling Time 55.5 Cycles |
| Rank 3 | Channel 4 (PA4), Sampling Time 55.5 Cycles |

### DMA Settings
- Add **ADC1** → DMA1 Channel 1.
- **Mode:** `Circular` (for continuous scan) or `Normal` (single sweep).
- **Data Width:** `Half Word` (ADC result is 12-bit; 16-bit transfers are correct).
- **Memory Increment:** Enabled.

> **Rank order matters.** Rank 1's result lands in `buffer[0]`, Rank 2 in `buffer[1]`, Rank 3 in `buffer[2]` — regardless of channel numbers.

## Generated Initialization (key excerpt)

```c
hadc1.Init.ScanConvMode          = ADC_SCAN_ENABLE;
hadc1.Init.ContinuousConvMode    = ENABLE;   // if circular free-run
hadc1.Init.NbrOfConversion       = 3;

// Rank 1
sConfig.Channel      = ADC_CHANNEL_0;
sConfig.Rank         = ADC_REGULAR_RANK_1;
sConfig.SamplingTime = ADC_SAMPLETIME_55CYCLES_5;
HAL_ADC_ConfigChannel(&hadc1, &sConfig);

// Rank 2
sConfig.Channel = ADC_CHANNEL_1;
sConfig.Rank    = ADC_REGULAR_RANK_2;
HAL_ADC_ConfigChannel(&hadc1, &sConfig);

// Rank 3
sConfig.Channel = ADC_CHANNEL_4;
sConfig.Rank    = ADC_REGULAR_RANK_3;
HAL_ADC_ConfigChannel(&hadc1, &sConfig);
```

## User Code

With DMA in **Circular** mode, call `HAL_ADC_Start_DMA()` once. The buffer is refreshed continuously by hardware[reference:2].

```c
#define N_CH  3
uint16_t adc_buf[N_CH];   // [0]=IN0, [1]=IN1, [2]=IN4

// In main(), before while(1):
HAL_ADCEx_Calibration_Start(&hadc1);
HAL_ADC_Start_DMA(&hadc1, (uint32_t *)adc_buf, N_CH);

// In while(1):
float v0 = adc_buf[0] * 3.3f / 4095.0f;
float v1 = adc_buf[1] * 3.3f / 4095.0f;
float v2 = adc_buf[2] * 3.3f / 4095.0f;
HAL_Delay(50);
```


## Optional — DMA Completion Callback

In Circular mode, `HAL_ADC_ConvCpltCallback` fires after **each full scan sequence** (not each channel). Use it to signal that a fresh triplet is ready.

```c
volatile uint8_t scan_done = 0;

void HAL_ADC_ConvCpltCallback(ADC_HandleTypeDef *hadc) {
    if (hadc->Instance == ADC1) {
        scan_done = 1;   // adc_buf[0..2] are fresh
    }
}
```


## Notes

- If DMA is `Normal` (not Circular), you must call `HAL_ADC_Start_DMA()` again after each sweep.
- A **100 nF capacitor** from each analog input to GND reduces noise, especially when scanning at high rate.
- Do **not** use pure polling with scan mode and continuous disabled unless you read `DR` twice immediately after end-of-sequence; DMA is the robust choice.